In [1]:
! pip install --upgrade ipywidgets

In [2]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel
import torch.nn.functional as F

In [3]:
# Load GPT-2 small
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)
model.eval()

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [4]:
# Example prompt
prompt = "ChatGPT explains things"
input_ids = tokenizer.encode(prompt, return_tensors="pt")

In [5]:
# Generation parameters
max_new_tokens = 10
temperature = 0.8
top_k = 50
top_p = 0.9

In [6]:
# Initialize sequence
generated_ids = input_ids.clone()
print("Initial prompt tokens:", tokenizer.convert_ids_to_tokens(input_ids[0].tolist()))

Initial prompt tokens: ['Chat', 'G', 'PT', 'Ġexplains', 'Ġthings']


In [7]:
# Step-by-step generation
for step in range(max_new_tokens):
    # Forward pass
    outputs = model(generated_ids)
    logits = outputs.logits[0, -1, :]  # logits for last token

    # Apply temperature
    logits = logits / temperature

    # Top-k filtering
    if top_k > 0:
        topk_vals, topk_indices = torch.topk(logits, top_k)
        mask = torch.ones_like(logits, dtype=torch.bool)
        mask[topk_indices] = False
        logits[mask] = -float("Inf")

    # Top-p (nucleus) filtering
    if top_p < 1.0:
        sorted_logits, sorted_indices = torch.sort(logits, descending=True)
        cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
        sorted_indices_to_remove = cumulative_probs > top_p
        # Shift mask to keep first token above threshold
        sorted_indices_to_remove[1:] = sorted_indices_to_remove[:-1].clone()
        sorted_indices_to_remove[0] = False
        indices_to_remove = sorted_indices[sorted_indices_to_remove]
        logits[indices_to_remove] = -float("Inf")

    # Compute probabilities
    probs = F.softmax(logits, dim=-1)

    # Sample next token
    next_token_id = torch.multinomial(probs, num_samples=1)
    generated_ids = torch.cat([generated_ids, next_token_id.unsqueeze(0)], dim=1)

    # Decode token
    next_token_word = tokenizer.decode(next_token_id)
    next_token_prob = probs[next_token_id].item()
    print(f"Step {step+1}: token ID={next_token_id.item()}, word='{next_token_word}', prob={next_token_prob:.4f}")

# Full generated text
full_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print("\nFull generated text:\n", full_text)

Step 1: token ID=287, word=' in', prob=0.0826
Step 2: token ID=517, word=' more', prob=0.4852
Step 3: token ID=3703, word=' detail', prob=1.0000
Step 4: token ID=287, word=' in', prob=0.1013
Step 5: token ID=674, word=' our', prob=0.1624
Step 6: token ID=649, word=' new', prob=0.1053
Step 7: token ID=1492, word=' book', prob=0.1922
Step 8: token ID=11, word=',', prob=0.5967
Step 9: token ID=383, word=' The', prob=0.4302
Step 10: token ID=3683, word=' Art', prob=0.0666

Full generated text:
 ChatGPT explains things in more detail in our new book, The Art
